In [ ]:
# [CELL 1] Imports, configuration, masks/index tables (precomputed once)
import os
import time
import math
import random
import itertools
from contextlib import nullcontext as _nullcontext
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import ExponentialLR
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)} | count: {torch.cuda.device_count()}")
torch.backends.cudnn.benchmark = True

# Tunable defaults; per-dataset overrides may set img_size / in_channels / model class.
DEFAULT_CONFIG = {
    'img_size':        18,     # downscaled image edge
    'grid_size':       6,      # 6x6 grid -> 36 cells, each 3x3 px
    'squares_to_keep': 4,      # squares the debate ends with
    'batch_size':      256,    # training batch size
    'epochs':          10,
    'lr':              1e-3,
    'lr_decay':        0.85,
    'eval_test_size':  10000,
    'eval_batch':      32,     # number of test images per debate-eval forward batch
    'eval_chunk_size': 60000,  # max rows per sparse_model forward (B*N_masks split)
    'in_channels':     1,      # MNIST/FashionMNIST -> 1, CIFAR10 -> 3
    'num_classes':     10,
    'eval_amp':        True,   # use torch.float16 autocast for sparse_model eval forward
    # ---- per-image probability dump + HuggingFace upload ----
    'dump_probs':      True,            # collect (n, N_masks, 10) probs per experiment
    'dump_n':          2000,            # number of test images to dump per experiment
    'dump_dtype':      'float16',       # 'float16' or 'bfloat16' (lossy) or 'float32'
    'dump_local_dir':  '/kaggle/working/dumps',
    'hf_repo_id':      None,            # e.g. 'username/mnistdebate-dumps'; None disables upload
    'hf_token':        None,            # if None, read from Kaggle Secrets or HF_TOKEN env var
    'hf_create_repo':  True,
    'hf_private':      True,
    'delete_local_after_upload': True,  # keeps /kaggle/working under quota
}


# --- HuggingFace upload helpers --------------------------------------------------
def _resolve_hf_token(explicit: str | None) -> str | None:
    if explicit:
        return explicit
    # Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient   # type: ignore
        try:
            return UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    except ImportError:
        pass
    return os.environ.get("HF_TOKEN")


def hf_upload(local_path: str, path_in_repo: str, config: dict) -> str | None:
    """Upload a single file to a HF dataset repo. Returns the remote URL or None."""
    repo_id = config.get('hf_repo_id')
    if not repo_id:
        print(f"  [hf] no hf_repo_id configured, skipping upload of {path_in_repo}")
        return None
    try:
        from huggingface_hub import HfApi
    except ImportError:
        print("  [hf] huggingface_hub not installed; run `pip install -q huggingface_hub`")
        return None
    token = _resolve_hf_token(config.get('hf_token'))
    if not token:
        print("  [hf] no token found (set HF_TOKEN env var or Kaggle Secret 'HF_TOKEN')")
        return None
    api = HfApi(token=token)
    if config.get('hf_create_repo', True):
        api.create_repo(repo_id=repo_id, repo_type='dataset',
                        private=config.get('hf_private', True), exist_ok=True)
    api.upload_file(path_or_fileobj=local_path, path_in_repo=path_in_repo,
                    repo_id=repo_id, repo_type='dataset')
    url = f"https://huggingface.co/datasets/{repo_id}/blob/main/{path_in_repo}"
    print(f"  [hf] uploaded -> {url}")
    return url


def build_mask_tables(img_size: int, grid_size: int, squares_to_keep: int, device: str):
    """Build ALL_MASKS plus the four minimax index tables used by debates.

    Returns a dict so that callers can swap configurations cleanly.
    """
    S, G, K = img_size, grid_size, squares_to_keep
    assert S % G == 0, "img_size must be divisible by grid_size"
    step = S // G
    total_cells = G * G

    combos = {k: list(itertools.combinations(range(total_cells), k)) for k in (1, 2, 3, K)}
    maps   = {k: {c: i for i, c in enumerate(combos[k])} for k in (2, 3, K)}

    # ALL masks of K squares, shape (N_masks, 1, S, S)
    n_masks = len(combos[K])
    all_masks = torch.zeros((n_masks, 1, S, S), device=device)
    for i, combo in enumerate(combos[K]):
        for cell in combo:
            r, c = cell // G, cell % G
            all_masks[i, 0, r*step:(r+1)*step, c*step:(c+1)*step] = 1.0

    # idx_3_to_4: from each size-3 state, the 33 size-4 children indices
    idx_3_to_4 = torch.zeros((len(combos[3]), total_cells - 3), dtype=torch.long)
    for i, s3 in enumerate(combos[3]):
        col = 0
        for x in range(total_cells):
            if x not in s3:
                idx_3_to_4[i, col] = maps[K][tuple(sorted(s3 + (x,)))]
                col += 1

    idx_2_to_3 = torch.zeros((len(combos[2]), total_cells - 2), dtype=torch.long)
    for i, s2 in enumerate(combos[2]):
        col = 0
        for x in range(total_cells):
            if x not in s2:
                idx_2_to_3[i, col] = maps[3][tuple(sorted(s2 + (x,)))]
                col += 1

    idx_1_to_2 = torch.zeros((len(combos[1]), total_cells - 1), dtype=torch.long)
    for i, s1 in enumerate(combos[1]):
        col = 0
        for x in range(total_cells):
            if x not in s1:
                idx_1_to_2[i, col] = maps[2][tuple(sorted(s1 + (x,)))]
                col += 1

    # idx_2_to_4: from a size-2 state, all (33 choose 2) = 561 ways B can pick its 2 squares.
    n_b_pairs = (total_cells - 2) * (total_cells - 3) // 2
    idx_2_to_4 = torch.zeros((len(combos[2]), n_b_pairs), dtype=torch.long)
    for i, s2 in enumerate(combos[2]):
        remaining = [x for x in range(total_cells) if x not in s2]
        for col, b_choices in enumerate(itertools.combinations(remaining, 2)):
            idx_2_to_4[i, col] = maps[K][tuple(sorted(s2 + b_choices))]

    return {
        'all_masks':  all_masks,
        'combos':     combos,
        'maps':       maps,
        'idx_3_to_4': idx_3_to_4.to(device),
        'idx_2_to_3': idx_2_to_3.to(device),
        'idx_1_to_2': idx_1_to_2.to(device),
        'idx_2_to_4': idx_2_to_4.to(device),
        'n_masks':    n_masks,
        'img_size':   S, 'grid_size': G, 'squares_to_keep': K, 'step': step,
    }


In [ ]:
# [CELL 2] Dataset preprocessing
#
# Loads a torchvision dataset, resizes to img_size, normalizes globally, and
# optionally applies per-class normalization (subtract per-class mean, add back
# global mean) so a linear classifier cannot separate classes by their mean
# direction. Returns (X_train, y_train, X_test, y_test) as GPU tensors so that
# the rest of the pipeline can skip DataLoader Python overhead.

DATASET_REGISTRY = {
    'mnist':        {'cls': torchvision.datasets.MNIST,        'channels': 1},
    'fashionmnist': {'cls': torchvision.datasets.FashionMNIST, 'channels': 1},
    'cifar10':      {'cls': torchvision.datasets.CIFAR10,      'channels': 3},
}


def _materialize(ds, img_size: int, channels: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Pull a torchvision dataset into a dense (N,C,S,S) float32 tensor + label tensor."""
    n = len(ds)
    X = torch.empty(n, channels, img_size, img_size, dtype=torch.float32)
    y = torch.empty(n, dtype=torch.long)
    resize = transforms.Resize((img_size, img_size), antialias=True)
    to_tensor = transforms.ToTensor()
    for i in range(n):
        img, lbl = ds[i]                                # PIL image
        img = to_tensor(resize(img))                    # (C, S, S) in [0,1]
        X[i] = img
        y[i] = lbl
    return X, y


def preprocess_dataset(name: str, img_size: int = 18,
                       normalize_per_class: bool = False,
                       data_root: str = './data',
                       device: str = DEVICE) -> dict:
    """Return a dict with train/test tensors and the in_channels for the model.

    normalize_per_class: when True, apply per-class ZCA whitening (with
        Tikhonov shrinkage on the covariance) using statistics computed on the
        combined train+test set. After this, every class has identical mean
        AND identical (identity) covariance, so a linear classifier on pixels
        gives ~10% by construction. Pixel-space (no PCA truncation), so the
        debate pipeline downstream still operates on 18x18 tensors.
    """
    if name not in DATASET_REGISTRY:
        raise ValueError(f"Unknown dataset: {name}")
    spec = DATASET_REGISTRY[name]
    channels = spec['channels']

    print(f"[preprocess] {name} | img_size={img_size} | per_class_norm={normalize_per_class}")
    train_raw = spec['cls'](root=data_root, train=True,  download=True)
    test_raw  = spec['cls'](root=data_root, train=False, download=True)

    Xtr, ytr = _materialize(train_raw, img_size, channels)
    Xte, yte = _materialize(test_raw,  img_size, channels)

    # Global standardization (per channel) computed on train set.
    mean = Xtr.mean(dim=(0, 2, 3), keepdim=True)
    std  = Xtr.std (dim=(0, 2, 3), keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr - mean) / std
    Xte = (Xte - mean) / std

    if normalize_per_class:
        # Per-class ZCA whitening on COMBINED train+test pixel statistics.
        #
        # For each class k:
        #   1. compute mu_k and Sigma_k from all (train + test) samples of class k
        #   2. regularize Sigma_reg = Sigma_k + lam * (tr Sigma_k / D) * I
        #      (ridge / Tikhonov shrinkage -- needed because at C*S*S = 324
        #      (MNIST) or 972 (CIFAR) dims we have <= 7k samples per class to
        #      estimate ~D^2/2 covariance parameters, so the raw sample cov is
        #      severely undersampled)
        #   3. ZCA whitening matrix W_k = Sigma_reg^{-1/2} via eigendecomposition
        #   4. apply x' = (x - mu_k) @ W_k + global_mean
        #
        # After this transform every class has mean ~ global_mean and
        # covariance ~ I, so first AND second class-conditional moments
        # coincide. A linear classifier therefore cannot separate the classes
        # (~10% by construction); a nonlinear classifier loses the simple
        # "detect an additive constant per class" shortcut that mean-centering
        # leaves wide open. Whitening is in pixel space (no PCA truncation) so
        # the debate visualization still operates on 18x18 tensors.
        lam_shrink = 0.05    # turn up if you see numerical instability on CIFAR
        D = channels * img_size * img_size
        X_all = torch.cat([Xtr, Xte], dim=0).view(-1, D).double()
        y_all = torch.cat([ytr, yte], dim=0)
        global_mean = X_all.mean(dim=0, keepdim=True)              # (1, D)
        eye_D = torch.eye(D, dtype=X_all.dtype)

        mus, Ws = [], []
        for k in range(10):
            Xk = X_all[y_all == k]
            mu_k = Xk.mean(dim=0, keepdim=True)
            Xc = Xk - mu_k
            Sigma_k = (Xc.T @ Xc) / (Xc.size(0) - 1)
            trace_per_d = Sigma_k.diagonal().mean()
            Sigma_reg = Sigma_k + lam_shrink * trace_per_d * eye_D
            eigvals, U = torch.linalg.eigh(Sigma_reg)
            eigvals = eigvals.clamp_min(eigvals.max() * 1e-8)
            W_k = (U * eigvals.rsqrt().unsqueeze(0)) @ U.T
            mus.append(mu_k)
            Ws.append(W_k)

        def _apply_whiten(X, y):
            Xflat = X.view(X.size(0), D).double()
            out = torch.empty_like(Xflat)
            for k in range(10):
                m = (y == k)
                if m.any():
                    out[m] = (Xflat[m] - mus[k]) @ Ws[k] + global_mean
            return out.float().view(X.size(0), channels, img_size, img_size)

        Xtr = _apply_whiten(Xtr, ytr)
        Xte = _apply_whiten(Xte, yte)

    return {
        'name':        name,
        'normalized':  normalize_per_class,
        'X_train':     Xtr.to(device),
        'y_train':     ytr.to(device),
        'X_test':      Xte.to(device),
        'y_test':      yte.to(device),
        'in_channels': channels,
        'img_size':    img_size,
    }


In [ ]:
# [CELL 3] Models
# Same architecture parameterized by in_channels. For CIFAR10 (3 channels) the
# first conv just absorbs more input planes; we also widen slightly because the
# task is much harder than MNIST at 18x18.

class SimpleCNN(nn.Module):
    def __init__(self, in_channels: int = 1, img_size: int = 18, num_classes: int = 10,
                 widen: int = 1):
        super().__init__()
        c1, c2 = 32 * widen, 64 * widen
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(c1, c2, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        # After two 2x2 pools at img_size=18 -> floor(18/2)=9 -> floor(9/2)=4 -> 4x4
        feat_hw = img_size // 4
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c2 * feat_hw * feat_hw, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def make_model(in_channels: int, img_size: int, num_classes: int = 10) -> nn.Module:
    widen = 2 if in_channels == 3 else 1     # bump capacity for CIFAR-like inputs
    return SimpleCNN(in_channels=in_channels, img_size=img_size,
                     num_classes=num_classes, widen=widen)


In [ ]:
# [CELL 4] Training
#
# generate_random_sparse_masks_vec replaces the per-image np.random.choice loop
# with a fully vectorized GPU op (torch.argsort over random scores, then
# repeat_interleave to expand grid cells to pixel blocks). On a T4 this drops
# training mask cost from ~0.4 ms/batch of Python work to a few microseconds.
#
# Training itself iterates over GPU-resident tensors with random index batches
# (no DataLoader workers needed -- the dataset already fits in VRAM).

def generate_random_sparse_masks_vec(batch_size: int, img_size: int, grid_size: int,
                                     squares_to_keep: int, device: str) -> torch.Tensor:
    """(B, 1, S, S) masks with `squares_to_keep` random grid cells set to 1."""
    B, S, G, K = batch_size, img_size, grid_size, squares_to_keep
    step = S // G
    cells = torch.rand(B, G * G, device=device).argsort(dim=1)[:, :K]  # (B, K)
    grid = torch.zeros(B, G * G, device=device)
    grid.scatter_(1, cells, 1.0)
    grid = grid.view(B, 1, G, G)
    masks = grid.repeat_interleave(step, dim=2).repeat_interleave(step, dim=3)
    return masks


def _iterate_minibatches(X: torch.Tensor, y: torch.Tensor, batch_size: int, shuffle: bool):
    n = X.size(0)
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    for start in range(0, n, batch_size):
        sel = idx[start:start + batch_size]
        yield X[sel], y[sel]


def train_classifier(data: dict, is_sparse: bool, config: dict,
                     mask_tables: dict, verbose: bool = True) -> tuple[nn.Module, float]:
    """Train one classifier (gold or sparse) end-to-end on a preprocessed dataset."""
    device = data['X_train'].device.type
    model = make_model(in_channels=data['in_channels'],
                       img_size=data['img_size'],
                       num_classes=config['num_classes']).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = ExponentialLR(optimizer, gamma=config['lr_decay'])
    criterion = nn.CrossEntropyLoss()

    S, G, K = data['img_size'], mask_tables['grid_size'], mask_tables['squares_to_keep']
    tag = 'sparse' if is_sparse else 'gold'

    if verbose:
        print(f"\n[train:{data['name']}/{'norm' if data['normalized'] else 'raw'}] "
              f"{tag} classifier -- {config['epochs']} epochs, "
              f"{data['X_train'].size(0)} train images")

    Xtr, ytr = data['X_train'], data['y_train']
    Xte, yte = data['X_test'],  data['y_test']
    n_train = Xtr.size(0)

    for epoch in range(config['epochs']):
        model.train()
        running = 0.0
        n_batches = 0
        for xb, yb in _iterate_minibatches(Xtr, ytr, config['batch_size'], shuffle=True):
            if is_sparse:
                masks = generate_random_sparse_masks_vec(xb.size(0), S, G, K, device)
                xb = xb * masks            # broadcasts over channels for 3-ch inputs
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item()
            n_batches += 1
        scheduler.step()
        if verbose:
            print(f"  epoch {epoch+1:02d}/{config['epochs']} | "
                  f"avg_loss={running / n_batches:.4f} | "
                  f"lr={scheduler.get_last_lr()[0]:.5f}")

    # Test accuracy
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in _iterate_minibatches(Xte, yte, 1024, shuffle=False):
            if is_sparse:
                masks = generate_random_sparse_masks_vec(xb.size(0), S, G, K, device)
                xb = xb * masks
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    acc = 100.0 * correct / total
    if verbose:
        print(f"  -> {tag} test accuracy: {acc:.2f}%")
    return model, acc


In [ ]:
# [CELL 5] Debate evaluation (batched + vectorized)
#
# Speed strategy:
#  * Process a batch of B test images simultaneously. Each image generates
#    N_masks (=58905 for 6x6 grid, 4 squares) masked versions; with B=32 that's
#    ~1.9M small forward passes per outer step but they go to the GPU as one
#    dense (B*N_masks, C, S, S) tensor split into chunks of eval_chunk_size.
#  * The minimax tables (idx_3_to_4, idx_2_to_3, idx_1_to_2, idx_2_to_4) are
#    used with an added batch dim via .unsqueeze + advanced indexing, so the
#    inner min/max collapses run entirely in CUDA.
#  * The per-image move-tracing argmaxes are cheap (microseconds) so we keep a
#    small Python loop over the batch for clarity.

def _trace_moves_for_image(i: int, true_label: int, probs: torch.Tensor,
                           V1: torch.Tensor, V2: torch.Tensor, V3: torch.Tensor,
                           V2_2turn: torch.Tensor,
                           mask_tables: dict, results: dict, plot_buf: dict,
                           img_for_plot: torch.Tensor):
    """Pull out the minimax-optimal moves for one image given precomputed values."""
    combos = mask_tables['combos']
    idx_3_to_4 = mask_tables['idx_3_to_4']
    idx_2_to_3 = mask_tables['idx_2_to_3']
    idx_1_to_2 = mask_tables['idx_1_to_2']
    idx_2_to_4 = mask_tables['idx_2_to_4']
    K = mask_tables['squares_to_keep']
    all_probs = probs                                                # (N_masks, 10)

    # ---- Sparse: random mask
    rand_idx = random.randint(0, all_probs.size(0) - 1)
    sparse_pred = all_probs[rand_idx].argmax().item()
    results['sparse'] += int(sparse_pred == true_label)
    _stash(plot_buf, 'sparse', sparse_pred == true_label, true_label, sparse_pred,
           img_for_plot, all_probs[rand_idx],
           [(sq, 'None', '') for sq in combos[K][rand_idx]])

    # ---- Consultancy: argmax confidence across masks
    best_cons = all_probs.max(dim=1).values.argmax().item()
    cons_pred = all_probs[best_cons].argmax().item()
    results['consultancy'] += int(cons_pred == true_label)
    _stash(plot_buf, 'consultancy', cons_pred == true_label, true_label, cons_pred,
           img_for_plot, all_probs[best_cons],
           [(sq, 'None', '') for sq in combos[K][best_cons]])

    # ---- 2-turn debate
    pred_2t = V2_2turn.max(dim=0).values.argmax().item()
    results['2_turn'] += int(pred_2t == true_label)
    s2_A_idx = V2_2turn[:, pred_2t].argmax().item()
    sqs_A = combos[2][s2_A_idx]
    b_choice_idx = all_probs[idx_2_to_4[s2_A_idx], pred_2t].argmin().item()
    mask_idx_2t = idx_2_to_4[s2_A_idx, b_choice_idx].item()
    sqs_B = tuple(set(combos[K][mask_idx_2t]) - set(sqs_A))
    results['judge_2_turn'] += int(all_probs[mask_idx_2t].argmax().item() == true_label)
    moves_2t = [(sqs_A[0], 'A', 1), (sqs_A[1], 'A', 1),
                (sqs_B[0], 'B', 2), (sqs_B[1], 'B', 2)]
    _stash(plot_buf, '2_turn', pred_2t == true_label, true_label, pred_2t,
           img_for_plot, all_probs[mask_idx_2t], moves_2t)

    # ---- 4-turn debate
    pred_4t = V1.max(dim=0).values.argmax().item()
    s1_A_idx = V1[:, pred_4t].argmax().item()
    sq1 = combos[1][s1_A_idx][0]
    s2_idx = idx_1_to_2[s1_A_idx, V2[idx_1_to_2[s1_A_idx], pred_4t].argmin().item()].item()
    sq2 = list(set(combos[2][s2_idx]) - {sq1})[0]
    s3_idx = idx_2_to_3[s2_idx, V3[idx_2_to_3[s2_idx], pred_4t].argmax().item()].item()
    sq3 = list(set(combos[3][s3_idx]) - set(combos[2][s2_idx]))[0]
    mask_idx_4t = idx_3_to_4[s3_idx, all_probs[idx_3_to_4[s3_idx], pred_4t].argmin().item()].item()
    sq4 = list(set(combos[K][mask_idx_4t]) - set(combos[3][s3_idx]))[0]
    results['4_turn'] += int(pred_4t == true_label)
    results['judge_4_turn'] += int(all_probs[mask_idx_4t].argmax().item() == true_label)
    moves_4t = [(sq1, 'A', 1), (sq2, 'B', 2), (sq3, 'A', 3), (sq4, 'B', 4)]
    _stash(plot_buf, '4_turn', pred_4t == true_label, true_label, pred_4t,
           img_for_plot, all_probs[mask_idx_4t], moves_4t)


def _stash(plot_buf, method, is_corr, true_lbl, pred_lbl, img_tensor, probs, moves):
    target = plot_buf[method]['correct'] if is_corr else plot_buf[method]['incorrect']
    if len(target) < 2:
        top_p, top_c = torch.topk(probs, 3)
        prob_text = [f"{c.item()}: {p.item()*100:.1f}%" for c, p in zip(top_c, top_p)]
        # img_tensor: (C, S, S) on GPU -> store as a HxWxC (or HxW for 1ch) numpy array
        if img_tensor.size(0) == 1:
            img_np = img_tensor[0].cpu().numpy()
        else:
            img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
            # rescale for matplotlib display
            lo, hi = img_np.min(), img_np.max()
            img_np = (img_np - lo) / max(hi - lo, 1e-8)
        target.append({'img': img_np, 'true': true_lbl, 'pred': pred_lbl,
                       'probs': prob_text, 'moves': moves})


def run_debates(gold_model: nn.Module, sparse_model: nn.Module, data: dict,
                config: dict, mask_tables: dict, verbose: bool = True) -> dict:
    """Run all 6 conditions across `eval_test_size` test images in batched fashion."""
    device = data['X_test'].device.type
    gold_model.eval(); sparse_model.eval()
    all_masks = mask_tables['all_masks']                                # (N, 1, S, S)
    idx_3_to_4 = mask_tables['idx_3_to_4']
    idx_2_to_3 = mask_tables['idx_2_to_3']
    idx_1_to_2 = mask_tables['idx_1_to_2']
    idx_2_to_4 = mask_tables['idx_2_to_4']
    N = mask_tables['n_masks']
    C = data['in_channels']
    S = mask_tables['img_size']

    n_eval = min(config['eval_test_size'], data['X_test'].size(0))
    X = data['X_test'][:n_eval]
    y = data['y_test'][:n_eval]

    plot_keys = ['gold', 'perfect', '4_turn', '2_turn', 'consultancy', 'sparse']
    keys = plot_keys + ['judge_2_turn', 'judge_4_turn']
    results = {k: 0 for k in keys}
    plot_buf = {k: {'correct': [], 'incorrect': []} for k in plot_keys}

    B = config['eval_batch']
    chunk = config['eval_chunk_size']
    use_amp = bool(config.get('eval_amp', False)) and device == 'cuda'
    amp_ctx = (torch.amp.autocast(device_type='cuda', dtype=torch.float16)
               if use_amp else _nullcontext())

    # --- Optional dump of per-image probabilities for offline reconstruction.
    # Pre-allocated as pinned CPU memory so we can copy probs out async per batch.
    dump_n = min(int(config.get('dump_n', 0) or 0), n_eval) if config.get('dump_probs') else 0
    if dump_n > 0:
        dump_dtype = getattr(torch, config.get('dump_dtype', 'float16'))
        dump_all_probs = torch.empty(dump_n, N, config['num_classes'],
                                      dtype=dump_dtype, pin_memory=(device == 'cuda'))
        dump_gold_probs = torch.empty(dump_n, config['num_classes'], dtype=dump_dtype)
        dump_true       = torch.empty(dump_n, dtype=torch.int16)
        dump_gold_pred  = torch.empty(dump_n, dtype=torch.int16)
    else:
        dump_all_probs = dump_gold_probs = dump_true = dump_gold_pred = None

    t0 = time.time()
    with torch.no_grad():
        # Pass 1: gold predictions for all eval images (cheap, single batched forward)
        gold_preds_all = torch.empty(n_eval, dtype=torch.long, device=device)
        gold_probs_all = torch.empty(n_eval, config['num_classes'], device=device)
        for s in range(0, n_eval, 1024):
            e = min(s + 1024, n_eval)
            with amp_ctx:
                logits = gold_model(X[s:e])
            p = torch.softmax(logits.float(), dim=1)
            gold_probs_all[s:e] = p
            gold_preds_all[s:e] = p.argmax(dim=1)
        results['gold'] = int((gold_preds_all == y).sum().item())

        # Pass 2: batched sparse forward + minimax + tracing
        iterator = tqdm(range(0, n_eval, B), desc=f"debate {data['name']}",
                        disable=not verbose)
        for s in iterator:
            e = min(s + B, n_eval)
            b = e - s
            imgs = X[s:e]                                       # (b, C, S, S)
            labels = y[s:e]
            gold_preds = gold_preds_all[s:e]

            # Stream masked images chunk-by-chunk so the full (b*N, C, S, S)
            # tensor is never materialized. Row r in [0, b*N) corresponds to
            # (image r // N, mask r % N).
            total_rows = b * N
            probs = torch.empty(total_rows, config['num_classes'], device=device)
            for cs in range(0, total_rows, chunk):
                ce = min(cs + chunk, total_rows)
                rows = torch.arange(cs, ce, device=device)
                img_idx = rows // N
                msk_idx = rows %  N
                chunk_imgs = imgs[img_idx] * all_masks[msk_idx]  # (ce-cs, C, S, S)
                with amp_ctx:
                    logits = sparse_model(chunk_imgs)
                probs[cs:ce] = torch.softmax(logits.float(), dim=1)
                del chunk_imgs, logits
            probs = probs.view(b, N, config['num_classes'])     # (b, N, 10)

            # Copy out to dump buffer (only the first `dump_n` images).
            if dump_n > 0 and s < dump_n:
                dump_e = min(e, dump_n)
                bd = dump_e - s
                dump_all_probs[s:dump_e].copy_(probs[:bd].to(dump_dtype),
                                                non_blocking=True)
                dump_gold_probs[s:dump_e].copy_(gold_probs_all[s:dump_e].to(dump_dtype))
                dump_true[s:dump_e]      = y[s:dump_e].cpu().to(torch.int16)
                dump_gold_pred[s:dump_e] = gold_preds_all[s:dump_e].cpu().to(torch.int16)

            # ---- Perfect: per-image argmax over masks of probability of gold_pred class
            perf_idx = probs.gather(2, gold_preds.view(b, 1, 1).expand(-1, N, 1)).squeeze(2).argmax(dim=1)  # (b,)
            perf_pred = probs[torch.arange(b, device=device), perf_idx].argmax(dim=1)
            results['perfect'] += int((perf_pred == labels).sum().item())

            # ---- Batched minimax values
            # V3: (b, |s3|, 33, 10) -> min over 33 -> (b, |s3|, 10)
            V3 = probs[:, idx_3_to_4].min(dim=2).values
            V2 = V3[:, idx_2_to_3].max(dim=2).values             # (b, |s2|, 10)
            V1 = V2[:, idx_1_to_2].min(dim=2).values             # (b, 36, 10)
            V2_2t = probs[:, idx_2_to_4].min(dim=2).values       # (b, |s2|, 10)

            # Per-image tracing (cheap, Python loop)
            for i in range(b):
                tl = labels[i].item()
                _stash(plot_buf, 'gold', gold_preds[i].item() == tl, tl,
                       gold_preds[i].item(), imgs[i], gold_probs_all[s + i], [])
                _stash(plot_buf, 'perfect', perf_pred[i].item() == tl, tl,
                       perf_pred[i].item(), imgs[i],
                       probs[i, perf_idx[i]],
                       [(sq, 'None', '') for sq in mask_tables['combos'][mask_tables['squares_to_keep']][perf_idx[i].item()]])
                _trace_moves_for_image(i, tl, probs[i], V1[i], V2[i], V3[i], V2_2t[i],
                                       mask_tables, results, plot_buf, imgs[i])

            del probs, V3, V2, V1, V2_2t

    dt = time.time() - t0
    n = n_eval
    summary = {k: 100.0 * results[k] / n for k in keys}
    summary['eval_time_sec'] = dt
    summary['n_eval'] = n

    if verbose:
        print(f"\n=== {data['name']} ({'per-class-norm' if data['normalized'] else 'raw'}) "
              f"-- debate eval in {dt:.1f}s ===")
        for k in plot_keys:
            print(f"  {k:<14s}: {summary[k]:6.2f}%")
        print(f"  judge_2_turn  : {summary['judge_2_turn']:6.2f}%")
        print(f"  judge_4_turn  : {summary['judge_4_turn']:6.2f}%")

    out = {'results': results, 'summary': summary, 'plot_data': plot_buf}

    if dump_n > 0:
        # Build the dump dict. Mask combos are tiny but useful for self-contained
        # reconstruction so we include them once per file (cheap, ~0.5MB at int16).
        K = mask_tables['squares_to_keep']
        combos4 = torch.tensor(mask_tables['combos'][K], dtype=torch.int16)  # (N, K)
        out['dump'] = {
            'dataset':           data['name'],
            'normalized':        bool(data['normalized']),
            'img_size':          mask_tables['img_size'],
            'grid_size':         mask_tables['grid_size'],
            'squares_to_keep':   K,
            'num_classes':       config['num_classes'],
            'n':                 dump_n,
            'all_probs':         dump_all_probs,   # (dump_n, N, num_classes)
            'gold_probs':        dump_gold_probs,  # (dump_n, num_classes)
            'true':              dump_true,        # (dump_n,)
            'gold_pred':         dump_gold_pred,   # (dump_n,)
            'combos4':           combos4,          # (N, K)
        }

    return out


In [ ]:
# [CELL 6] Plotting
# Given the plot_data returned by run_debates, render the four-panel figure per
# method (2 correct + 2 incorrect examples) and save under experiment_plots/.

def plot_results(plot_data: dict, dataset_tag: str, out_dir: str = "experiment_plots",
                 grid_size: int = 6, step: int = 3, show: bool = False) -> list[str]:
    os.makedirs(out_dir, exist_ok=True)
    saved = []

    def draw(ax, d, title_prefix):
        img = d['img']
        moves = d['moves']
        is_color = (img.ndim == 3)
        if len(moves) > 0:
            base = np.zeros_like(img)
            for sq, _, _ in moves:
                r, c = sq // grid_size, sq % grid_size
                base[r*step:(r+1)*step, c*step:(c+1)*step] = img[r*step:(r+1)*step, c*step:(c+1)*step]
            disp = np.maximum(base, img * 0.15)
        else:
            disp = img
        if is_color:
            ax.imshow(np.clip(disp, 0, 1))
        else:
            ax.imshow(disp, cmap='gray', vmin=img.min(), vmax=img.max())
        for sq, player, turn in moves:
            r, c = sq // grid_size, sq % grid_size
            y, x = r * step, c * step
            color = {'A': '#ff3333', 'B': '#3399ff'}.get(player, '#66ff66')
            ax.add_patch(patches.Rectangle((x-0.5, y-0.5), step, step,
                                            fill=False, edgecolor=color, linewidth=2))
            if turn != '':
                ax.text(x + step/2, y + step/2, str(turn), color=color,
                        ha='center', va='center', fontweight='black', fontsize=14)
        ax.set_title(f"{title_prefix}\nTruth: {d['true']} | Selected: {d['pred']}",
                     fontsize=11, pad=10)
        ax.axis('off')
        ax.text(0.5, -0.05, " | ".join(d['probs']),
                transform=ax.transAxes, ha='center', va='top', fontsize=10)

    for method, data in plot_data.items():
        fig, axs = plt.subplots(2, 2, figsize=(10, 10))
        fig.suptitle(f"{dataset_tag} -- Method: {method.upper().replace('_', ' ')}",
                     fontsize=16, fontweight='bold', y=0.98)
        examples = data['correct'][:2] + data['incorrect'][:2]
        titles = ["Correct 1", "Correct 2", "Incorrect 1", "Incorrect 2"]
        for i, ax in enumerate(axs.flatten()):
            if i < len(examples):
                draw(ax, examples[i], titles[i])
            else:
                ax.axis('off')
        plt.tight_layout(rect=[0, 0.05, 1, 0.95])
        path = os.path.join(out_dir, f"{dataset_tag}_{method}.png")
        plt.savefig(path, dpi=150, bbox_inches='tight')
        if show:
            plt.show()
        else:
            plt.close(fig)
        saved.append(path)
    return saved


In [ ]:
# [CELL 7] Run the full sweep
#
# Six experiments: {MNIST, FashionMNIST, CIFAR10} x {raw, per-class normalized}.
# The mask/index tables only depend on img_size+grid_size+K, so they can be
# built once (in_channels does not affect them: masks broadcast over channels).

CONFIG = dict(DEFAULT_CONFIG)
# Set your HF dataset repo here, e.g. 'your-username/mnistdebate-dumps'.
# Leave as None to skip uploading (dumps will still be written under dump_local_dir
# unless you also set 'dump_probs' to False).
CONFIG['hf_repo_id'] = None     # <-- EDIT ME, e.g. 'username/mnistdebate-dumps'

print("Building mask + minimax index tables...")
MASK_TABLES = build_mask_tables(CONFIG['img_size'], CONFIG['grid_size'],
                                CONFIG['squares_to_keep'], DEVICE)
print(f"  n_masks = {MASK_TABLES['n_masks']}")

EXPERIMENTS = [
    ('mnist',        False),
    ('mnist',        True),
    ('fashionmnist', False),
    ('fashionmnist', True),
    ('cifar10',      False),
    ('cifar10',      True),
]

if CONFIG.get('dump_probs'):
    os.makedirs(CONFIG['dump_local_dir'], exist_ok=True)

all_summaries = {}

for name, per_class in EXPERIMENTS:
    tag = f"{name}{'_normalized' if per_class else ''}"
    print(f"\n############### {tag} ###############")

    data = preprocess_dataset(name, img_size=CONFIG['img_size'],
                              normalize_per_class=per_class, device=DEVICE)
    cfg = dict(CONFIG)
    cfg['in_channels'] = data['in_channels']

    gold_model,   gold_acc   = train_classifier(data, is_sparse=False, config=cfg,
                                                mask_tables=MASK_TABLES)
    sparse_model, sparse_acc = train_classifier(data, is_sparse=True,  config=cfg,
                                                mask_tables=MASK_TABLES)

    out = run_debates(gold_model, sparse_model, data, cfg, MASK_TABLES)
    out['summary']['gold_base_test_acc']   = gold_acc
    out['summary']['sparse_base_test_acc'] = sparse_acc
    all_summaries[tag] = out['summary']

    plot_results(out['plot_data'], dataset_tag=tag,
                 grid_size=MASK_TABLES['grid_size'], step=MASK_TABLES['step'])

    # ---- Save + upload per-image probability dump for this experiment ----
    if 'dump' in out:
        # Decorate dump with the achieved accuracies so the offline file is self-describing.
        out['dump']['summary'] = {k: float(v) for k, v in out['summary'].items()}
        dump_path = os.path.join(CONFIG['dump_local_dir'], f"{tag}.pt")
        torch.save(out['dump'], dump_path)
        size_mb = os.path.getsize(dump_path) / 1e6
        print(f"  [dump] wrote {dump_path} ({size_mb:.1f} MB)")

        url = hf_upload(local_path=dump_path, path_in_repo=f"{tag}.pt", config=CONFIG)
        if url and CONFIG.get('delete_local_after_upload', True):
            os.remove(dump_path)
            print(f"  [dump] removed local copy to keep /kaggle/working under quota")

    # free GPU memory before the next dataset
    del data, gold_model, sparse_model, out
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print("\n\n================ FINAL SUMMARY ================")
header = f"{'experiment':<24s}" + "".join(f"{k:>12s}" for k in
    ['gold', 'perfect', '4_turn', '2_turn', 'consultancy', 'sparse',
     'judge_2t', 'judge_4t'])
print(header)
for tag, s in all_summaries.items():
    row = f"{tag:<24s}" + "".join(
        f"{s[k]:>11.2f}%" for k in
        ['gold', 'perfect', '4_turn', '2_turn', 'consultancy', 'sparse',
         'judge_2_turn', 'judge_4_turn'])
    print(row)


## Offline reconstruction from a HuggingFace dump

Each experiment uploads a single file `{dataset}_{raw|normalized}.pt` to the
configured HF dataset repo. To analyse only one experiment without re-running
the notebook, download just that file (the other 5 stay on HF). All six files
together at the default `dump_n=2000` are ~14 GB; one file is ~2.4 GB.

```python
from huggingface_hub import hf_hub_download
import torch, itertools

# pick exactly the experiment you want
path = hf_hub_download(
    repo_id="your-username/mnistdebate-dumps",
    filename="mnist_normalized.pt",      # or fashionmnist.pt, cifar10_normalized.pt, ...
    repo_type="dataset",
)
dump = torch.load(path, map_location="cpu", weights_only=False)

probs    = dump["all_probs"].float()      # (n, 58905, 10) -- sparse classifier probs per mask
gold_p   = dump["gold_probs"].float()     # (n, 10)        -- full-image classifier probs
y_true   = dump["true"].long()            # (n,)
gold_pred= dump["gold_pred"].long()       # (n,)
combos4  = dump["combos4"].long()         # (58905, 4)     -- cell ids of the 4 squares per mask
N, K = combos4.shape                       # 58905, 4

# Example 1: reproduce consultancy accuracy (best of 58905 masks by confidence).
best_idx = probs.max(dim=2).values.argmax(dim=1)              # (n,)
cons_pred = probs[torch.arange(len(probs)), best_idx].argmax(dim=1)
print("consultancy:", (cons_pred == y_true).float().mean().item())

# Example 2: k-th-best consultancy. Sort masks by max confidence, pick rank k.
def kth_best_consultancy(probs, k):
    conf = probs.max(dim=2).values                            # (n, N)
    rank_idx = conf.argsort(dim=1, descending=True)[:, k]      # (n,)
    sel = probs[torch.arange(len(probs)), rank_idx]
    return sel.argmax(dim=1)

for k in [0, 1, 5, 50, 500]:
    pred = kth_best_consultancy(probs, k)
    print(f"  k={k:4d}: {(pred == y_true).float().mean().item():.3f}")

# Example 3: reproduce the 4-turn minimax (you'll need to rebuild the index
# tables from combos4 itertools, see build_mask_tables() in the notebook).
```
